# SPADE M3: Token-Level Rejection Analysis and Profiling

This research notebook analyzes execution traces from **Milestone M2** baseline runs (B0, B0-S, B1, B2) to understand **where, why, and under which conditions** speculative token proposals are rejected.

### Important Methodological Rules:
1. **No Fabricated Data**: This notebook operates exclusively on real experimental run directories generated by `scripts/run_baselines.py`.
2. **Distributional Preservation**: In exact speculative sampling (B2), the output distribution matches the target model exactly. Speculative rejection reflects drafter divergence from the target distribution, **not** factual incorrectness.
3. **Timing Transparency**: Timing metrics distinguish full-prefix recomputation from optimized KV-cache decoding.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from spade.analysis.metrics import acceptance_summary
from spade.analysis.plots import analyze_runs
from spade.analysis.report import generate_experiment_report

# Point to actual completed run directories under outputs/runs/
# e.g. RUN_DIRS = [Path('../outputs/runs/m2-b1-k4'), Path('../outputs/runs/m2-b2-k4')]
RUN_DIRS = list(Path('../outputs/runs').glob('m2-*'))
OUT_DIR = Path('../outputs/analysis/m3_report')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Discovered {len(RUN_DIRS)} run directories:")
for r in RUN_DIRS:
    print(f" - {r.name}")

## 1. Load Traces and Compute Global Acceptance Diagnostics

We load token-level traces and compute:
- Observed acceptance rates (token-weighted, round-weighted, claim-weighted)
- Expected acceptance probability $\sum_v \min(p(v), q(v)) = 1 - \text{TV}(p, q)$
- Mean target $p(x)$ vs draft $q(x)$ proposal probabilities
- Mean accepted and committed tokens per speculative round

In [ ]:
if RUN_DIRS:
    report = generate_experiment_report(RUN_DIRS, OUT_DIR)
    summary = report.get("acceptance_summary", {})
    print(json.dumps(summary, indent=2))
else:
    print("No completed runs found yet in outputs/runs/. Follow docs/SERVER_RUNBOOK.md to execute M2 baselines on the GPU server.")

## 2. Token Category Acceptance Breakdown

Token proposals are categorized into transparent diagnostic groups:
- `numeric`: Numeric digits and values
- `date_related`: Months, years, date formats
- `punctuation`: Syntactic and punctuation markers
- `citation`: Evidence citation markers (e.g. `[E1]`, `E2`)
- `potential_named_entity`: Capitalized entity tokens
- `other`: General vocabulary

In [ ]:
cat_csv = OUT_DIR / "plots" / "acceptance_by_category.csv"
if cat_csv.exists():
    df_cat = pd.read_csv(cat_csv)
    display(df_cat)
    plt.figure(figsize=(8, 4))
    plt.bar(df_cat["category"], df_cat["acceptance_rate"], color="#1f77b4")
    plt.ylabel("Acceptance Rate")
    plt.title("Acceptance Rate by Token Category")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 1.0)
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("Category acceptance CSV not available yet.")

## 3. Rejection Position Distribution

Analyzes the proposal index (0 to $K-1$) at which the target model rejected the draft proposal. A high concentration at position 0 indicates immediate draft divergence.

In [ ]:
pos_csv = OUT_DIR / "plots" / "rejection_positions.csv"
if pos_csv.exists():
    df_pos = pd.read_csv(pos_csv)
    display(df_pos)
    plt.figure(figsize=(7, 4))
    plt.bar(df_pos["position"], df_pos["count"], color="#d62728")
    plt.xlabel("Rejection Proposal Position (0-indexed)")
    plt.ylabel("Number of Rounds")
    plt.title("Histogram of Rejection Positions")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("Rejection positions CSV not available yet.")

## 4. Latency Decomposition and Speedup by Draft Length

Examines decoding time partitioned into:
- `draft_generation_s`: Small model autoregressive generation of $K$ tokens
- `target_verification_s`: Target model parallel verification forward pass
- `correction_s`: Sampling from residual distribution / bookkeeping

In [ ]:
lat_csv = OUT_DIR / "plots" / "latency_breakdown.csv"
speedup_csv = OUT_DIR / "plots" / "speedup_comparison.csv"
if lat_csv.exists():
    df_lat = pd.read_csv(lat_csv)
    display(df_lat.head(10))
if speedup_csv.exists():
    df_speedup = pd.read_csv(speedup_csv)
    display(df_speedup)
else:
    print("Latency / speedup tables not available yet.")

## 5. Token Acceptance vs. Context Length

Tracks whether long evidence contexts degrade draft model acceptance rate.

In [ ]:
ctx_csv = OUT_DIR / "plots" / "acceptance_by_context_length.csv"
if ctx_csv.exists():
    df_ctx = pd.read_csv(ctx_csv)
    display(df_ctx)
    plt.figure(figsize=(8, 4))
    plt.plot(df_ctx["prefix_tokens_bin_start"], df_ctx["acceptance_rate"], marker="o", color="#2ca02c")
    plt.xlabel("Context Prefix Length (Tokens)")
    plt.ylabel("Observed Acceptance Rate")
    plt.title("Acceptance Rate vs. Context Length")
    plt.ylim(0, 1.0)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("Context length acceptance CSV not available yet.")

## 6. Distribution of $p(x) / q(x)$ Probability Ratios

Visualizes the proposal quality under the target model. Values $\ge 1$ are accepted with probability 1.

In [ ]:
pq_csv = OUT_DIR / "plots" / "pq_ratio_diagnostics.csv"
if pq_csv.exists():
    df_pq = pd.read_csv(pq_csv)
    display(df_pq.describe())
else:
    print("p/q diagnostics CSV not available yet.")

## 7. Example Token-Level Trace Walkthrough

Displays a real generation trace showing sequential proposal verification, rejection positions, and residual corrections.

In [ ]:
trace_csv = OUT_DIR / "plots" / "example_token_trace.csv"
if trace_csv.exists():
    df_trace = pd.read_csv(trace_csv)
    display(df_trace[["round_id", "absolute_token_position", "token_id", "token_text", "status", "target_probability", "draft_probability", "acceptance_probability"]].head(25))
else:
    print("Example token trace CSV not available yet.")